# Inference

In [70]:
import sys, os
sys.path.append("D:/Github/Deepfake_Detection")
import torch
import torch.nn.functional as F
import numpy as np
from src.utils.arguments import get_args
from src.base_pipeline import BasePipeline
from src.base_dataset import BaselineDataset
from torch.utils.data import DataLoader
import logging
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.models import efficientnet_b1, inception_v3
from torchvision import transforms
import timm

In [71]:
device = 'cuda'
class_names = ['real', 'fake']
ckpt_path = f'D:/Github/Deepfake_Detection/Deepfake_Detection_stage2/checkpoint/best.pt'
checkpoint = torch.load(ckpt_path, map_location=device)
logger = logging.getLogger(__name__)
transformation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
json_file = 'D:/Github/Deepfake_Detection/data/image_labels_test.json'
model = timm.create_model('resnet50_clip.openai', pretrained=True)
baseline_dataset = BaselineDataset(json_file=json_file, model = model, transformation=transformation)

In [76]:
def load_pipeline(checkpoint_path=ckpt_path, device=device, num_classes=2):
    """Load the pipeline directly from checkpoint"""
    logger.info(f"Loading pipeline from {checkpoint_path}")
    
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        backbone_model = efficientnet_b1(pretrained=False)
        
        pipeline = BasePipeline(
            model=backbone_model,
            args=None,
            device=device,
            in_features=2048,
            num_label=num_classes
        ).to(device)
        
        if 'state_dict' in checkpoint:
            pipeline_state_dict = {}
            for key, value in checkpoint['state_dict'].items():
                if key.startswith('pipeline.'):
                    # Remove 'pipeline.' prefix
                    new_key = key[9:]  # Remove 'pipeline.'
                    pipeline_state_dict[new_key] = value
            
            # Load the state dict
            pipeline.load_state_dict(pipeline_state_dict)
            logger.info("Pipeline loaded from Lightning checkpoint")
        else:
            # Direct pipeline checkpoint
            pipeline.load_state_dict(checkpoint)
            logger.info("Pipeline loaded from direct checkpoint")
            
        pipeline.eval()
    except Exception as e:
        logger.error(f"Failed to load pipeline: {e}")
        raise
    
    return pipeline

pipeline = load_pipeline()

c:\Users\Admin\miniconda3\envs\visualization\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\miniconda3\envs\visualization\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [ ]:
def predict_single(pipeline, input_tensors, device):
    with torch.no_grad():
        if len(input_tensors.shape) == 3:
            input_tensors = input_tensors.unsqueeze(0)

        input_tensors = input_tensors.to(device)
        logits = pipeline.forward_pipeline(input_tensors)
        probability = F.softmax(input=logits, dim=1)
        predicted_class = torch.argmax(input = probability, dim=1)
        confidence = torch.max(input = probability, dim=1)[0]

    return {
        "logits": logits,
        "probability": probability,
        "predicted_class": predicted_class,
        "confidence": confidence
    }

In [74]:
sample_input = torch.randn(2048, 7, 7)
result = predict_single(pipeline, sample_input, device)

print(f"Predicted class: {class_names[result['predicted_class'].item()]}")
print(f"Confidence: {result['confidence'].item():.4f}")

Predicted class: real
Confidence: 0.5660


In [75]:
def predicted_batch(pipeline, dataloader, device):
    all_probabilities = []
    all_prediction = []
    all_confidence = []
    all_labels = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if isinstance(batch, (list,tuple)) and len(batch)==2:
                inputs, labels = batch
            else:
                inputs = batch
                labels = None
            inputs = inputs.to(device)

        logits = pipeline.forward_pipeline(inputs)
        probability = F.softmax(input=logits, dim=1)
        predicted_class = torch.argmax(input = probability, dim=1)
        confidence = torch.max(input = probability, dim=1)[0]

        all_prediction.extend(predicted_class.cpu().numpy())
        all_probabilities.extend(probability.cpu().numpy())
        all_confidence.extend(confidence.cpu().numpy())

        if labels is not None:
            all_labels.extend(labels.cpu().numpy())

    results = {
        'predictions': np.array(all_predictions),
        'probabilities': np.array(all_probabilities),
        'confidences': np.array(all_confidences)
    }
    
    if all_labels:
        results['labels'] = np.array(all_labels)
    return results

In [81]:
def evaluate_model(results, class_names=None):
    """Evaluate model performance"""
    if 'labels' not in results:
        logger.warning("No ground truth labels available for evaluation")
        return None
    
    predictions = results['predictions']
    labels = results['labels']
    confidences = results['confidences']
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    
    logger.info(f"Overall Accuracy: {accuracy:.4f}")
    logger.info(f"Average Confidence: {np.mean(confidences):.4f}")
    
    # Classification report
    if class_names is None:
        class_names = [f"Class_{i}" for i in range(len(np.unique(labels)))]
    
    report = classification_report(labels, predictions, target_names=class_names)
    logger.info("Classification Report:")
    logger.info(f"\n{report}")
    
    # Confusion matrix
    cm = confusion_matrix(labels, predictions)
    
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {
        'accuracy': accuracy,
        'classification_report': report,
        'confusion_matrix': cm,
        'avg_confidence': np.mean(confidences)
    }

In [82]:
dataloader = torch.utils.data.DataLoader(
    baseline_dataset,
    batch_size=20,
    num_workers=16,
    shuffle=False
)
results = predicted_batch(pipeline, dataloader, device='cuda')

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\Admin\miniconda3\envs\visualization\Lib\site-packages\torch\utils\data\_utils\worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\visualization\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\miniconda3\envs\visualization\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "D:\Github/Deepfake_Detection\src\base_dataset.py", line 51, in __getitem__
    image = Image.open(img_path).convert('RGB')
            ^^^^^^^^^^^^^^^^^^^^
  Fil

In [ ]:
if 'labels' in results:
    metrics = evaluate_model(results, class_names)
    print(f"Test Accuracy: {metrics['accuracy']:.4f}")